In [ ]:
import os

os.environ["HF_HOME"] = "/path/to/hf"
os.environ["HF_HUB_CACHE"] = "/path/to/hf"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = "/path/to/hf"

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from glob import glob
from sentence_transformers import CrossEncoder

models = glob("../helmBenchmark/*.pkl")
model = models[0]
df = pd.read_pickle(model)

In [ ]:
def get_labels(score_list):
    l=[]
    for scores in score_list: #scores represents the list of scores in a row
        if len(scores) > 0: #not empty (f1 / bleu score exist)
            temp=[]
            for score in scores:
                label_mapping = ['contradiction', 'entailment', 'neutral']
                labels = [label_mapping[score_max] for score_max in score.argmax(axis=1)] 
                temp.append(labels)
            l.append(temp)
        else:
            l.append([])
    return l

def get_entailment_model_decision(labels):
    Vee, Eee, Ven, Een = [], [], [], []
    
    for line in labels:
        if len(line) > 0:
    
            vee_flag = True
            eee_flag = False
            ven_flag = True
            een_flag = False
    
            for element in line:
                if element != ['entailment', 'entailment']:
                    vee_flag = False
                if element == ['entailment', 'entailment']:
                    eee_flag = True
                if element not in [['entailment', 'entailment'], ['entailment', 'neutral'], ['neutral', 'entailment']]:
                    ven_flag = False
                if element in [['entailment', 'entailment'], ['entailment', 'neutral'], ['neutral', 'entailment']]:
                    een_flag = True
    
            Vee.append(1 if vee_flag else 0)
            Eee.append(1 if eee_flag else 0)
            Ven.append(1 if ven_flag else 0)
            Een.append(1 if een_flag else 0)
        else:
            Vee.append(np.nan)
            Eee.append(np.nan)
            Ven.append(np.nan)
            Een.append(np.nan)

    return Vee, Eee, Ven, Een

NLI = CrossEncoder('cross-encoder/nli-deberta-v3-large', device='cuda')

In [ ]:
import re
reg = r'\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}'

math = df.loc[df['task'] == 'math']
decision = []
for _, row in math.iterrows():
    pred = row['predicted_text']
    ref = row['references'][0]['output']['text']
    try:
        pred_box = re.search(reg, pred).group(1)
        ref_box = re.search(reg, ref).group(1)
        score = NLI.predict([(pred_box, ref_box), (ref_box, pred_box)])
        labels = get_labels([[score]])
        decision.append(get_entailment_model_decision(labels))
    except: #wrong formatting
        decision.append(([0], [0], [0], [0])) #for each entailment mode

In [4]:
mode = [row[3][0] for row in decision]
equal_mask = math["stats_math_equiv_chain_of_thought"] == mode
num_equal = equal_mask.sum()

print("Total equal:", num_equal)
print(len(math))

Total equal: 423
437


In [ ]:
wrong_indices = math.index[~equal_mask]
print(wrong_indices.tolist())

In [6]:
i = 711437
pred = df.loc[i]['predicted_text']
ref = df.loc[i]['references'][0]['output']['text']
pred_box = re.search(reg, pred).group(1)
ref_box = re.search(reg, ref).group(1)
score = NLI.predict([(pred_box, ref_box), (ref_box, pred_box)])
pred, ref, df.loc[i]['stats_math_equiv_chain_of_thought'], pred_box, ref_box, get_labels([[score]])

(' There are four possible outcomes for the election: Dan and Freddie win, Dan and Bernie win, Donald and Freddie win, and Donald and Bernie win. The probability of each of these outcomes is $\\frac{1}{2}\\times\\frac{1}{2}=\\frac{1}{4}$. Since we want the probability of Dan and Freddie winning, we only need to consider the first outcome. Therefore, the probability that both Dan and Freddie win is $\\boxed{\\frac{1}{4}}$.\n',
 'The probability that Dan wins is $\\frac12$. The probability that Freddie wins is also $\\frac12$. Therefore, the probability that both win is $\\frac12 \\cdot \\frac12 =\\boxed{\\frac14}$.',
 np.float64(1.0),
 '\\frac{1}{4}',
 '\\frac14',
 [[['contradiction', 'contradiction']]])

In [ ]:
openbook = df.loc[df['task'] == 'commonsense']
decision = []

label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F'}
for _, row in openbook.iterrows():
    pred = row['predicted_text']
    ref = ''
    for r in range(len(row['references'])):
        if row['references'][r]['tags'] == ['correct']:
            ref = label[r]
    try:
        score = NLI.predict([(pred, ref), (ref, pred)])
        labels = get_labels([[score]])
        decision.append(get_entailment_model_decision(labels))
    except: #wrong formatting
        print(pred, ref)
        decision.append(([0], [0], [0], [0])) #for each entailment mode

In [8]:
mode = [row[3][0] for row in decision]
equal_mask = openbook["stats_exact_match"] == mode
num_equal = equal_mask.sum()

print("Total equal:", num_equal)
print(len(openbook))

Total equal: 500
500


In [11]:
openbook

,instance_id,train_trial_index_x,predicted_text,base64_images,stats_num_prompt_tokens,stats_num_output_tokens,stats_inference_runtime,stats_num_train_instances,stats_prompt_truncated,stats_bleu_4,...,mapped_output,stats_exact_match,stats_final_number_exact_match,stats_math_equiv_chain_of_thought,stats_f1_score,similarity,Vee,Eee,Ven,Een
706346,id4957,0,B,[],275.0,1.0,0.357264,5.0,0.0,NaN,...,quit eating lunch out,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706347,id4958,0,A,[],236.0,1.0,0.554964,5.0,0.0,NaN,...,a marsh,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706348,id4959,0,C,[],227.0,1.0,0.404248,5.0,0.0,NaN,...,bunnies,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706349,id4960,0,C,[],270.0,1.0,0.498870,5.0,0.0,NaN,...,parts may break the concrete,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706350,id4961,0,B,[],233.0,1.0,0.285029,5.0,0.0,NaN,...,a power station,0.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
706841,id5452,0,A,[],251.0,1.0,0.321487,5.0,0.0,NaN,...,scalds,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706842,id5453,0,C,[],247.0,1.0,0.317688,5.0,0.0,NaN,...,water is bubbling from applied warmth,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706843,id5454,0,C,[],247.0,1.0,0.588757,5.0,0.0,NaN,...,leads to less sick people,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
706844,id5455,0,B,[],255.0,1.0,0.319496,5.0,0.0,NaN,...,tiny lifeforms in dirt,1.0,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN
